# Registration ID / BulkID identification, and a cleaned taxonomy for everything else

Two questions over the 11,442 tickets in `data/reg_id/reg_id_last3_months.xlsx`:

1. **Which tickets are Registration ID / BulkID / remote-credential requests?**
   A regex already exists for this, written in Tableau against `[Incident Subject]`.
2. **What is everything else about?** Using a cleaned-up version of the taxonomy hand-built
   in `data/reg_id/reg_id_manually tagged.xlsx`.

## What the profiling found, and why this notebook is shaped the way it is

**There is no subject line in the new data.** The export carries `INCIDENT_REF, Incident_ID,
Channel, Chat_Start_Date, Conversation` and nothing else. `Conversation` is the whole
speaker-labelled thread, customer and agent together.

**Run against the whole thread, the regex is high-recall and low-precision.** Scored against the
7,082 manual tags: recall 97% (272 of 281 `BulkIDs*` rows), precision **31%**. Of the 607 false
positives, **356 are `Complimentary access request`** - the customer asked for free access and the
*agent* answered by issuing registration IDs. The regex matched the answer, not the question.

**So the regex runs on the customer's turns only.** Every conversation in the export carries
`Customer:` / `Agent:` markers; 652 of them (5.7%) have no customer turn at all and fall back to
the full text. Matching customer segments alone takes 1,190 hits down to 603, and the adjudicator
in section 8 sees no agent reply for the other 94% - it cannot match on text the customer did not
write.

**The two files share no ticket IDs** (`Incident_ID` intersect `Document Id` = 0). The tagged sheet
is a separate labelled corpus, used here as the gate-4 evaluation set, not joined on.

## What this notebook does not tell you

The gate-4 figure in section 9 is scored against the tagged corpus, whose `Document` column is
mostly **the agent reply alone** (1.8% contain `Customer:`) where the production data is a full
thread. Different text shape, so treat that score as indicative of the taxonomy's separability, not
as the accuracy of this run. Nothing in the deliverable is a measured accuracy figure.

## Shape of the run

| | input | rows | what it decides |
|---|---|---|---|
| `regid_adjudicate` | customer turns of the regex-flagged tickets | ~602 | is the customer really asking for Reg/Bulk IDs? |
| `regid_taxonomy` | full conversation, every ticket | 11,442 | one label from the cleaned taxonomy |

They are independent - no routing between them. The final label is the adjudicated BulkIDs verdict
where the adjudicator says yes, and the taxonomy label everywhere else.


## 0. Configuration — edit this cell


In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# The system Python has pandas built against NumPy 1.x under NumPy 2.x and
# raises "_ARRAY_API not found" on import. Fail now rather than mid-run.
assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
TICKETS_XLSX = BASE / 'data' / 'reg_id' / 'reg_id_last3_months.xlsx'
# Named "last3_months" but it spans 2025-09-15 to 2026-09-20. Nothing here
# filters on date; the whole file is classified.
TAGGED_XLSX  = BASE / 'data' / 'reg_id' / 'reg_id_manually tagged.xlsx'
SHEET_TAGGED = 'raw data tagged'   # 7,082 hand-tagged rows, 102 distinct labels
SHEET_CANON  = 'Distinct'          # the tagger's own 25-row consolidation
SHEET_SYN    = 'Categories'        # same 25 rows plus a Synonyms column

for _p in (TICKETS_XLSX, TAGGED_XLSX):
    assert _p.exists(), f'missing input: {_p}'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'reg_id_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_regid'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'
DATE_COL = 'Chat_Start_Date'

# -- Module A: adjudicate the regex hits ----------------------------------
MOD_A    = 'regid_adjudicate'
PREFIX_A = 'RIDA'
STUB_A   = 'rida'

# -- Module T: the cleaned taxonomy, every ticket -------------------------
MOD_T    = 'regid_taxonomy'
PREFIX_T = 'RIDT'
STUB_T   = 'ridt'

ALL_MODULES     = [MOD_A, MOD_T]
MODULE_PREFIXES = {MOD_A: PREFIX_A, MOD_T: PREFIX_T}

# -- Preprocessing --------------------------------------------------------
SHORT_CHAR_THRESHOLD = 40      # payload chars below which a ticket carries no information
TRUNC_CAP            = 16_000  # per-conversation ceiling sent to the model
TRUNC_HEAD           = 4_000
TRUNC_TAIL           = 11_900  # 11,900 not 12,000: the truncation marker counts toward the cap

# -- LLM tuning -----------------------------------------------------------
# Inherited unchanged from the ScienceDirect and ClinicalKey runs, which
# completed ~34,000 tickets between them without a retry storm.
TUNING = {
    'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4,
    'retries': 5, 'max_tokens': 8192, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
    'temperature': 0.0,
}
# Which of the above change what the model returns, and so must invalidate a
# checkpoint when edited. batch_size and max_chars are in the list because they
# decide which records share a request, and a record batched with different
# neighbours is read in a different context. Throughput settings are excluded, so
# re-tuning concurrency or retries costs nothing.
OUTPUT_AFFECTING = ('temperature', 'max_tokens', 'batch_size', 'max_chars')
assert set(OUTPUT_AFFECTING) <= set(TUNING), (
    f'OUTPUT_AFFECTING names keys absent from TUNING: {set(OUTPUT_AFFECTING) - set(TUNING)}')

# -- Outputs --------------------------------------------------------------
INTER_DIR = BASE / 'runs' / 'intermediate'
SMOKE_DIR = BASE / 'runs' / 'smoke_regid'
OUT_DIR   = BASE / 'runs' / 'output'
DOCS_DIR  = BASE / 'docs'
for _d in (INTER_DIR, SMOKE_DIR, OUT_DIR, DOCS_DIR, DEFN_XLSX.parent, BASE / 'profiles'):
    _d.mkdir(parents=True, exist_ok=True)

INPUT_A    = INTER_DIR / 'regid_adjudicate_input.csv'   # customer turns, flagged rows only
INPUT_T    = INTER_DIR / 'regid_taxonomy_input.csv'     # full conversation, every row
PROFILE_A  = BASE / 'profiles' / 'reg_id_2026_adjudicate.json'
PROFILE_T  = BASE / 'profiles' / 'reg_id_2026_taxonomy.json'
TAX_MAP_MD = DOCS_DIR / 'REGID_2026_taxonomy_map.md'
FINAL_XLSX = OUT_DIR / 'reg_id_2026_classified.xlsx'
FINAL_CSV  = OUT_DIR / 'reg_id_2026_classified_full_text.csv'

# rca run prints U+2192 in its help text and log lines. Under the Windows
# console default (cp1252) that raises UnicodeEncodeError and kills the subprocess,
# so every child process gets UTF-8 forced on.
SUBPROCESS_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}

print(f'Project root : {BASE}')
print(f'Tickets      : {TICKETS_XLSX.relative_to(BASE)}')
print(f'Tagged set   : {TAGGED_XLSX.relative_to(BASE)}  [{SHEET_TAGGED}]')
print(f'Definition   : {DEFN_XLSX.relative_to(BASE)}')
print(f'Modules      : {ALL_MODULES}')


## 0b. Shared helpers

`norm_label` is the one function everything else depends on. Labels arrive from three different
sheets of the same workbook and are joined on their text, so a trailing space or a non-breaking
space in one of them silently splits a category in two. Normalise once, join on the result.


In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '\u200b\u200c\u200d\ufeff'))


def norm_label(value) -> str:
    """Normalise a taxonomy label to a stable join key, preserving its wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)          # delete before NFKC
    text = unicodedata.normalize('NFKC', text)  # folds U+00A0 -> ' '
    return re.sub(r'\s+', ' ', text).strip()


def leaf_labels(module: str) -> dict:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


print('Helpers defined.')


In [ ]:
_CASES = [
    ('Not registered / signed in to access org content ',                 # trailing space:
     'Not registered / signed in to access org content'),                 # a real duplicate label
    ('IP\xa0management', 'IP management'),                                # non-breaking space
    ('Account  deletion', 'Account deletion'),                            # doubled space
    ('BulkIDs\u200b', 'BulkIDs'),                                         # zero-width space
    (None, ''), ('nan', ''), ('   ', ''),
]
for raw, want in _CASES:
    got = norm_label(raw)
    assert got == want, f'norm_label({raw!r}) -> {got!r}, expected {want!r}'
print(f'norm_label: all {len(_CASES)} cases pass.')


## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook only,
so any module not in that workbook is dropped. The config holds one product at a time. Whatever is
live right now gets backed up here first, with the command to put it back written alongside.


In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_live = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
_live_modules = [m for m in _live['modules'] if m not in ALL_MODULES]

_backed_up = []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *[TAXONOMY_DIR / f'{m}.json' for m in _live_modules]]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():                 # never clobber an earlier backup
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}')
for _n in _backed_up:
    print(f'  {_n}')
print(f'\nModules in the live config right now: {list(_live["modules"])}')

# Which definition workbook rebuilds them. Matched on the module name prefix
# rather than hard-coded, so this stays right whichever product was last live.
_DEFN_OF = {'sd_ac_': 'sd_2026_accessing_content_definition.xlsx',
            'sd_': 'sd_2026_pipeline_definition.xlsx',
            'csd_': 'csd_2026_pipeline_definition.xlsx',
            'leapspace': 'leapspace_2026_pipeline_definition.xlsx'}
_defn = next((v for k, v in _DEFN_OF.items()
              if any(m.startswith(k) for m in _live_modules)), None)
_restore = (f'rca build --input definitions/{_defn} --force-all'
            if _defn else '(no known definition workbook rebuilds these - see generated/)')

# Written once and never rewritten. The second time this cell runs the live
# config is already this notebook's own, so _live_modules is empty - overwriting
# would replace the only record of what was there before with "(none)".
_restore_txt = CONFIG_BACKUP / 'RESTORE.txt'
if _restore_txt.exists():
    print(f'\n{_restore_txt.relative_to(BASE)} already records the pre-run state; '
          f'left untouched:\n')
    print('    ' + _restore_txt.read_text(encoding='utf-8').replace('\n', '\n    '))
else:
    _restore_txt.write_text(
        'The modules that were live in generated/pipeline_config.json before this notebook '
        'ran:\n'
        f'    {", ".join(_live_modules) or "(none)"}\n\n'
        'To put them back:\n\n'
        f'    {_restore}\n\n'
        'Their outputs are already saved under runs/output/.\n'
        'The config holds one product at a time; restoring them drops the regid_* modules.\n',
        encoding='utf-8')
    print(f'\nTo restore afterwards:\n    {_restore}')


## 1. The Registration ID / BulkID regex

Ported verbatim from the Tableau calculated field. Two alternations:

**Registration / Bulk ID terms** — `(registration|bulk|reg)` optionally followed by `user` or
`account`, then `id`/`ids`/`id's`/`identifier(s)`/`number(s)`/`code(s)`. The `[^a-z0-9]*` joins
absorb every separator seen in the data, so `Registration ID`, `RegistrationID`, `Bulk-ID`,
`Bulk_IDs`, `RegID`, `Bulk User ID` and `Registration Account ID` all match.

**Remote credential terms** — `remote` optionally followed by `user`/`access`/`admin`/`login`, then
`credentials` or `creds`. `creden?t(ial|ail|al)s?` is what catches the misspellings
`credetials`, `credentails` and `credentals`.

The two changes from Tableau: it runs on the customer's turns rather than a subject line (see the
header), and Python's `re` needs no escaping change - the pattern is character-for-character the
string Tableau was given.


In [ ]:
# Character-for-character the Tableau REGEXP_MATCH pattern.
REGID_PATTERN = (
    r"(registration|bulk|reg)[^a-z0-9]*((user|account)[^a-z0-9]*)?"
    r"(id['\u2019]?s?|identifiers?|numbers?|codes?)"
    r"|remote[^a-z0-9]*((user|access|admin|login)[^a-z0-9]*)?"
    r"(creden?t(ial|ail|al)s?|creds?)")
REGID_RX = re.compile(REGID_PATTERN)          # applied to lower-cased text, as LOWER() did

# Two forms of the same pattern. split() needs the capture group to tell the
# speakers apart; str.contains() warns about a group it will not use, so it
# gets the non-capturing twin.
_SPEAKER_RE  = re.compile(r'(?mi)^\s*(customer|agent)(?:\s+notes)?\s*:')
_SPEAKER_ANY = re.compile(r'(?mi)^\s*(?:customer|agent)(?:\s+notes)?\s*:')
_CUSTOMER_RE = re.compile(r'(?mi)^\s*customer(?:\s+notes)?\s*:')


def customer_turns(text) -> str:
    """Just the customer's half of a thread.

    re.split with one capture group returns [before, speaker, segment, speaker,
    segment, ...], so the speakers sit at odd indices and their text one further
    on. Every conversation in this export carries markers, but ~5.7% contain no
    CUSTOMER turn - an agent-logged note or a phone summary. For those the whole
    text is returned rather than an empty string, which would silently drop the
    ticket out of the regex population.
    """
    text = '' if text is None or (isinstance(text, float) and pd.isna(text)) else str(text)
    parts = _SPEAKER_RE.split(text)
    segs = [parts[i + 1] for i in range(1, len(parts), 2)
            if parts[i].lower().startswith('customer')]
    return ' '.join(s.strip() for s in segs).strip() if segs else text.strip()


def regid_match(text) -> str:
    """The literal substring the regex matched, or '' - for eyeballing the hits."""
    m = REGID_RX.search(str(text).lower())
    return m.group(0) if m else ''


### Self-test — every term the brief listed must match, and the near-misses must not

The negatives matter more than the positives here. `regional` starts with `reg`, and
`bulk order number` is a real phrase in these tickets that has nothing to do with credentials -
both are checked to confirm the pattern is not simply matching everything.


In [ ]:
_MATCH = [
    # Registration / Bulk ID terms, exactly as listed in the brief
    'Registration ID', 'RegistrationID', 'Registration IDs',
    'Bulk ID', 'Bulk-ID', 'Bulk_IDs',
    'RegID', 'Reg ID',
    'Bulk User ID',
    'Registration Account ID',
    # variants ending in identifier / number / code
    'registration identifier', 'bulk id number', 'reg identifiers', 'registration code',
    "registration ID's", 'registration ID\u2019s',
    # Remote credential terms
    'Remote Credentials', 'Remote User Credentials', 'Remote Access Credentials',
    'Remote Admin Credentials', 'Remote Login Credentials', 'Remote Creds', 'remote cred',
    # misspellings
    'remote credetials', 'remote credentails', 'remote credentals',
    # in context
    'could you also provide me with 5 remote access registration ids.',
]
_NO_MATCH = [
    'regional office', 'regular access', 'bulky attachment',
    'remote working', 'credentials for SSO',      # 'credentials' without 'remote'
    'please register', 'your registration is complete',
]
for _s in _MATCH:
    assert REGID_RX.search(_s.lower()), f'should match but does not: {_s!r}'
for _s in _NO_MATCH:
    assert not REGID_RX.search(_s.lower()), f'should NOT match but does: {_s!r}'

# customer_turns must actually drop the agent's half - this is the whole reason
# precision goes from 31% to something usable.
_thread = ('Customer: We need free access for our students.\n'
           'Agent: Here are your registration ids: 123, 456.\n'
           'Customer: Thank you.')
assert 'registration ids' not in customer_turns(_thread).lower(), 'agent turn leaked through'
assert not REGID_RX.search(customer_turns(_thread).lower()), 'comp-access ticket still flags'
assert customer_turns('no markers here at all') == 'no markers here at all', 'fallback broken'

print(f'Regex self-test: {len(_MATCH)} positives, {len(_NO_MATCH)} negatives, '
      f'3 speaker-split cases - all pass.')


## 2. The cleaned taxonomy

The tagged sheet holds 102 distinct labels over 7,082 rows: about twenty real categories and an
eighty-label tail of singletons, misspellings (`Unabled to login`, `Remove uses`,
`Not instituonal linked`) and one-off phrasings.

The tagger had already started consolidating: the `Distinct` and `Categories` tabs hold 25
hand-written Self-Service Categories with descriptions and synonym lists. Those are the backbone.
Six of the 25 rows are notes rather than categories and are dropped; the categories carrying real
volume in the raw 102 that the Distinct tab has no home for are added.


In [ ]:
canon_tab = pd.read_excel(TAGGED_XLSX, sheet_name=SHEET_CANON, dtype=str)
syn_tab   = pd.read_excel(TAGGED_XLSX, sheet_name=SHEET_SYN, dtype=str)
canon_tab.columns = [norm_label(c) for c in canon_tab.columns]
syn_tab.columns   = [norm_label(c) for c in syn_tab.columns]

_LABEL_COL = 'Self-Service Categories'
# Rows of the Distinct tab that are notes to self, not categories. Asserted
# present so a re-export that renames or removes one fails here rather than
# smuggling 'Key:' into the taxonomy as a label.
NOT_CATEGORIES = [
    'Key:',
    'Take it as it is for Account center',
    'Split those related to users for Acenter',
    'Non Self-Service Categories',
]
_seen = set(canon_tab[_LABEL_COL].map(norm_label))
for _n in NOT_CATEGORIES:
    assert _n in _seen, f'expected note row missing from {SHEET_CANON}: {_n!r} - re-check the tab'

# Keyed by the tagger's own label, which is not the cleaned label - re-keyed in
# the cell after next, once TAGGER_LABEL_TO_CANONICAL exists.
TAGGER_DESC_RAW = {
    norm_label(r[_LABEL_COL]): norm_label(r.get('Decription'))
    for _, r in canon_tab.iterrows()
    if norm_label(r[_LABEL_COL]) and norm_label(r[_LABEL_COL]) not in NOT_CATEGORIES}
TAGGER_SYN_RAW = {
    norm_label(r[_LABEL_COL]): norm_label(r.get('Synonyms'))
    for _, r in syn_tab.iterrows()
    if norm_label(r[_LABEL_COL]) and norm_label(r[_LABEL_COL]) not in NOT_CATEGORIES}

print(f'{SHEET_CANON} tab: {len(canon_tab)} rows, {len(TAGGER_DESC_RAW)} usable categories, '
      f'{sum(1 for v in TAGGER_DESC_RAW.values() if v)} with a description, '
      f'{sum(1 for v in TAGGER_SYN_RAW.values() if v)} with synonyms')
for _k in TAGGER_DESC_RAW:
    print(f'  {_k}')


In [ ]:
# ---------------------------------------------------------------------------
# The cleaned taxonomy.
#
# Backbone: the 'Distinct' tab of the manually tagged workbook - 25 rows the
# tagger curated by hand, with descriptions. Six of those rows are not
# categories at all ('Key:', 'Take it as it is for Account center', ...) and are
# dropped in the next cell against the workbook itself, so a re-export that
# renames them fails loudly instead of smuggling a note in as a label.
#
# Additions: every category carrying real volume in the 102 raw labels that the
# Distinct tab has no home for - 'Complimentary access request' (379 rows),
# 'Private email access' (390), password reset (353), and so on. Without them
# the highest-volume part of the tail would have nowhere to land.
#
# ORDER IS LOAD-BEARING. rca build assigns taxonomy codes by row order,
# so RIDT1.1 is whatever sits first here. Append, never insert, or every code in
# a previously written deliverable shifts by one.
# ---------------------------------------------------------------------------
CANONICAL = {
    'BulkIDs / Registration IDs': (
        'The customer is asking for Registration IDs, RegIDs, BulkIDs or remote credentials - '
        'to be created, added, deleted, exchanged, extended or explained. Also covers questions '
        'about whether an organisation has them and how many are left. The defining feature is '
        'that the CUSTOMER is requesting or asking about the IDs themselves.'),
    'IP management': (
        'An admin asking to add, delete, update, remove, whitelist or check IP addresses or IP '
        'ranges on their account, or reporting that access fails because they are outside the '
        'registered IP range, or asking who added an IP. Includes reports of abnormal use from '
        'an IP and blocked IPs.'),
    'Account deletion / remove users': (
        'A request to delete a user account or to remove, unlink or replace a user on an '
        'account - normally because the person has left the organisation, or because of content '
        'theft. Covers a person asking for their own account to be deleted and an admin asking '
        'on behalf of somebody else, plus removing a user association to an organisation or '
        'department.'),
    'SSO / Shibboleth configuration': (
        'An admin asking for federated access to be set up, changed or disabled for their '
        'organisation: SSO, Shibboleth (Shib), adding an IdP or Identity Provider, OpenAthens, '
        'SAML attributes, metadata, certificate renewal. Use this for configuration requests AND '
        'for a user reporting that SSO or Shibboleth sign-in is not working.'),
    'Domain management': (
        'Adding, removing, deleting or setting up an email domain or subdomain on an account. '
        'Includes Email Domain Confirmation, Email Institutional Association (EIA), Self Manage '
        'Remote Access (SelfManRA), and the case where a user cannot be associated because their '
        'email address does not match the registered domain.'),
    'Change or add email address': (
        'A request to change or add the email address on somebody else\'s account - moving users '
        'from one account to another, or updating the email a user is registered under. The '
        'request is made FOR A THIRD PERSON, not for the person contacting.'),
    'Admin access': (
        'Granting, removing or asking about administrative rights: Admin Tool access, Admin '
        'Console access, admin privileges, setting somebody up as an admin, removing admin '
        'rights. About the ADMIN ROLE itself, not about what an admin is trying to do.'),
    'Merge accounts': (
        'A user asking to merge two email addresses, profiles or accounts into one, to add an '
        'old email to a current account, or to transfer saved data from one account to another. '
        'Also covers a user who has ended up with two accounts and does not know why.'),
    'Create customer account or organisation': (
        'Creating an end-user account on behalf of somebody, or creating/adding an organisation '
        'record in the internal systems (CHIEF / Phoenix): create ECR, set up a super account. '
        'Raised internally or by an admin, not a self-service registration.'),
    'Create department': (
        'Creating a department under an existing account in Account Center or Phoenix.'),
    'Correct organisation record (CHIEF/Phoenix)': (
        'Correcting an existing organisation record: change the account or super-account name, '
        'update the legal name for an ECR, clean up an ECR or its departments, or an admin '
        'reporting their institution name is displayed wrongly.'),
    'Usage reporting': (
        'Requests about usage reports: generating or downloading a usage report, SUSHI, COUNTER, '
        'reporting on which users consumed what.'),
    'Change logo': (
        'A request to add, change or remove the organisation logo shown in the product.'),
    'Block user': (
        'A request to block or bar a specific user from access - typically an unauthorised user '
        'or one suspected of misuse. Distinct from deletion: the account stays, access stops.'),
    'Course access / enrolment': (
        'A student or instructor unable to access, join or enrol on a course, or asking for '
        'enrolment to be changed.'),
    'Access or activation codes': (
        'Anything about a redemption, access or activation code supplied to a customer: a code '
        'that will not redeem, a request for a code, or a question about what a code covers. '
        'NOT the same as BulkIDs - a BulkID is a set of credentials issued to an organisation, '
        'an activation code unlocks a single product or title.'),
    'Delete data': (
        'A request to delete personal data or account history rather than the account itself - '
        'typically framed as a privacy or GDPR request.'),
    'Access or reset link expired': (
        'A link the customer was sent has expired or stopped working before they could use it: '
        'an institutional access link, a WAYFless link, an email confirmation link or a password '
        'reset link. The complaint is about the LINK, not about the password itself.'),
    'Password reset': (
        'The user cannot sign in because of their password and needs it reset or changed: '
        'forgotten password, a reset request, a password that will not be accepted. If the '
        'complaint is that the reset LINK expired, use the link category instead.'),
    'Other account settings': (
        'Changing other details on a profile or account that no other category covers: name, '
        'salutation, address, notification or communication preferences, display settings. The '
        'tagger used this as a catch-all for account changes, so prefer a specific category '
        'whenever one fits.'),
    'Complimentary access request': (
        'A request for free, trial, courtesy or goodwill access to content or a product - for an '
        'individual, a group or a country programme. The ASK is for access to be granted at no '
        'charge. If the agent happens to fulfil it by issuing registration IDs, that does NOT '
        'make it a BulkIDs request; classify on what the customer asked for.'),
    'Add affiliation': (
        'A user asking to be linked or affiliated to their institution or organisation, or '
        'reporting that they are not associated with it and therefore cannot get access.'),
    'Private email access': (
        'A user trying to reach organisational content while signed in with a personal or '
        'private email address (gmail, hotmail, a non-institutional domain), or asking whether '
        'they may use one.'),
    'Not registered or signed in': (
        'The user has access available to them but is not registered, or is not signed in, and '
        'so sees paywalled or limited content. The fix is to register or sign in, not a '
        'configuration change.'),
    'Registering an account': (
        'A user asking how to register, unable to complete registration, or asking for an '
        'individual account to be set up. About the ACT of registering, not about affiliation '
        'once registered.'),
    'Sign-in or browser issue': (
        'Sign-in failing for a technical reason rather than a password or entitlement one: '
        'cookies, cache, browser compatibility, a login loop, a session that will not hold. '
        'Also the case where the user reports the problem resolved itself.'),
    'HTTP or platform error': (
        'The user reports a numbered or named platform error - Error 710, Error 702, a 403, a '
        '404, a server error page - rather than describing a functional request.'),
    'WAYFless URL request': (
        'A request for a WAYFless URL, or for an existing WAYFless link to be reissued, '
        'corrected or explained.'),
    'Organisation not subscribed / not entitled': (
        'Access fails because the organisation does not hold a subscription or entitlement to '
        'the content requested, or holds only a limited one. Nothing is misconfigured; the '
        'entitlement is simply absent.'),
    'Agreement or entitlement info request': (
        'A question ABOUT an agreement or entitlement rather than a fault: what is covered, '
        'when it runs to, which titles are included, what an organisation is entitled to.'),
    'Royalty link info request': (
        'A question about royalty links, royalty statements or the royalty portal.'),
    'Billing or invoicing': (
        'A question or problem about an invoice, a payment, a purchase order, a refund or a '
        'renewal charge.'),
    'General information request': (
        'A general question about how something works or where to find it, that no more '
        'specific category covers, and that is not a fault or a change request.'),
    'Not enough info': (
        'The ticket does not say enough to classify it: an empty or near-empty body, an '
        'auto-reply, a bare acknowledgement, an attachment with no text, or a thread whose '
        'substance is missing. Use this rather than guessing.'),
}

TAGGER_LABEL_TO_CANONICAL = {
    'IP management': 'IP management',
    'Account deletion (source: admin)': 'Account deletion / remove users',
    'SSO configuration / info': 'SSO / Shibboleth configuration',
    'BulkIDs': 'BulkIDs / Registration IDs',
    'Change / Add email (source: admin)': 'Change or add email address',
    'Domain management': 'Domain management',
    'Admin access': 'Admin access',
    'Merge accounts': 'Merge accounts',
    'Create customer account (source: Relx internal)': 'Create customer account or organisation',
    'Create dpt (A.C. OR Phoenix)': 'Create department',
    'Correct account in CHIEF/Phoenix': 'Correct organisation record (CHIEF/Phoenix)',
    'Usage reporting': 'Usage reporting',
    'Change logo': 'Change logo',
    'Block user': 'Block user',
    'Course access / enrollment': 'Course access / enrolment',
    'Activation Code': 'Access or activation codes',
    'Delete data': 'Delete data',
    'Forgot password/Change reset': 'Password reset',
    # norm_label folds the ellipsis to three dots under NFKC, so the key carries
    # three dots and not U+2026.
    'Update other profile info (name/address/salutation...)': 'Other account settings',
}

# Every one of the 102 raw labels in the 'raw data tagged' sheet, mapped onto the
# canonical set above. Asserted total in the next cell - a raw label that is not
# in this dict raises rather than being dropped.
#
# Misspellings in the keys ('Unabled to login', 'Remove uses', 'unathorized',
# 'instituonal') are the tagger's and are reproduced exactly; they are join keys,
# not prose.
RAW_TO_CANONICAL = {
    # -- BulkIDs -----------------------------------------------------------
    'BulkIDs': 'BulkIDs / Registration IDs',
    'BulkIDs issue': 'BulkIDs / Registration IDs',
    'BulkIDs steps': 'BulkIDs / Registration IDs',
    'Registration mandatory management': 'BulkIDs / Registration IDs',
    # -- IP ----------------------------------------------------------------
    'IP management': 'IP management',
    'IP information request': 'IP management',
    'IP blocked': 'IP management',
    'IP management-abnormal use': 'IP management',
    'Not within IP range': 'IP management',
    'Not within IP address': 'IP management',
    'Not within IP': 'IP management',
    # Tagged as both; IP is the actionable half and the larger category.
    'IP management / Domain management': 'IP management',
    # -- Deletion / user removal -------------------------------------------
    'Account deletion': 'Account deletion / remove users',
    'Account deletion - admin': 'Account deletion / remove users',
    'Delete account in WAYF': 'Account deletion / remove users',
    'Remove users': 'Account deletion / remove users',
    'Remove user': 'Account deletion / remove users',
    'Remove uses': 'Account deletion / remove users',
    'Removing users': 'Account deletion / remove users',
    'Remove institutional association': 'Account deletion / remove users',
    'unlink wrong users account': 'Account deletion / remove users',
    'Handle users': 'Account deletion / remove users',
    'Update users in account': 'Account deletion / remove users',
    # -- SSO ----------------------------------------------------------------
    'SSO/Shibboleth': 'SSO / Shibboleth configuration',
    'SSO/Shibboleth request of new link': 'SSO / Shibboleth configuration',
    'Issue with SSO': 'SSO / Shibboleth configuration',
    'SSO login': 'SSO / Shibboleth configuration',
    'No password (SSO)': 'SSO / Shibboleth configuration',
    # -- Domain -------------------------------------------------------------
    'Domain management': 'Domain management',
    'Domain management issue': 'Domain management',
    'Domain management: info provided': 'Domain management',
    'Domain steps': 'Domain management',
    'Email domain steps': 'Domain management',
    'Email domain setup': 'Domain management',
    'Email domain management': 'Domain management',
    "Email doesn't match domain": 'Domain management',
    "email doesn't match domain": 'Domain management',
    # -- Email change --------------------------------------------------------
    'Changing emails': 'Change or add email address',
    # -- Admin ---------------------------------------------------------------
    'Admin access': 'Admin access',
    'Admin Tool access request': 'Admin access',
    'Admin support': 'Admin access',
    'Add admin': 'Admin access',
    'Remove admin access': 'Admin access',
    # -- Merge ---------------------------------------------------------------
    'Merge accounts': 'Merge accounts',
    '2 acccounts (not known)': 'Merge accounts',
    # -- Org records ---------------------------------------------------------
    'Create account - admin': 'Create customer account or organisation',
    'Super account request': 'Create customer account or organisation',
    'New organization change/setup': 'Correct organisation record (CHIEF/Phoenix)',
    'Update legal name': 'Correct organisation record (CHIEF/Phoenix)',
    'Change account name': 'Correct organisation record (CHIEF/Phoenix)',
    # -- Reporting / logo / block --------------------------------------------
    'Usage reporting': 'Usage reporting',
    'User reporting': 'Usage reporting',
    'Changing logo': 'Change logo',
    'Block unathorized user': 'Block user',
    # -- Codes ----------------------------------------------------------------
    'Access codes': 'Access or activation codes',
    # -- Passwords -------------------------------------------------------------
    'Pw reset': 'Password reset',
    'Forgot password': 'Password reset',
    'Pw reset link expiring': 'Password reset',
    'Link expired': 'Access or reset link expired',
    'link expired': 'Access or reset link expired',
    # -- Settings ----------------------------------------------------------------
    'Other account settings': 'Other account settings',
    # -- Comp access --------------------------------------------------------------
    'Complimentary access request': 'Complimentary access request',
    'Ukraine access': 'Complimentary access request',
    # -- Affiliation ----------------------------------------------------------------
    'Add affiliation': 'Add affiliation',
    'Not associated to institution': 'Add affiliation',
    'Not instituonal linked': 'Add affiliation',
    'not institutional association': 'Add affiliation',
    'Unable to login. No institutional association': 'Add affiliation',
    # -- Private email ------------------------------------------------------------
    'Private email access': 'Private email access',
    'Using private email to access org content': 'Private email access',
    # -- Not registered -------------------------------------------------------------
    'Not registered / signed in to access org content': 'Not registered or signed in',
    'Not registered / signed in to access org content (EIA)': 'Not registered or signed in',
    # -- Registration ----------------------------------------------------------------
    'Registering account': 'Registering an account',
    'Help with registering an account': 'Registering an account',
    'Individual account setup': 'Registering an account',
    'No account created': 'Registering an account',
    'Cannot register other account (cookies)': 'Registering an account',
    # -- Sign-in / browser ----------------------------------------------------------
    'Unable to login': 'Sign-in or browser issue',
    'Unabled to login': 'Sign-in or browser issue',
    'Unable to sign in [cookies]': 'Sign-in or browser issue',
    'Clear cache and cookies': 'Sign-in or browser issue',
    'Login issue solved on its own': 'Sign-in or browser issue',
    # -- Errors -----------------------------------------------------------------------
    'HTTP errors': 'HTTP or platform error',
    'Error 710': 'HTTP or platform error',
    'Error 702': 'HTTP or platform error',
    # -- WAYFless ------------------------------------------------------------------------
    'WAYFless URL request': 'WAYFless URL request',
    # -- Entitlement ----------------------------------------------------------------------
    'Org does not subscribe to content': 'Organisation not subscribed / not entitled',
    'Org no subscription': 'Organisation not subscribed / not entitled',
    'Not org subscription': 'Organisation not subscribed / not entitled',
    'Not entitled': 'Organisation not subscribed / not entitled',
    'No entitled': 'Organisation not subscribed / not entitled',
    'Limited view of author profiles': 'Organisation not subscribed / not entitled',
    'Agreement info request': 'Agreement or entitlement info request',
    'Entitlement info request': 'Agreement or entitlement info request',
    'Agreement info request. Not entitled': 'Agreement or entitlement info request',
    'Royalty link info request': 'Royalty link info request',
    'Billing or invoicing inquiries': 'Billing or invoicing',
    # -- Catch-alls --------------------------------------------------------------------------
    'General information request': 'General information request',
    'Info on how to access': 'General information request',
    'Scopus': 'General information request',
    'Not enough info': 'Not enough info',
}


### Prove the mapping is total

Every one of the 102 raw labels must resolve, every target must exist, and no canonical category
may be unreachable except the three carried over from the tagger's own list that happen to have no
rows yet. A raw label that is not in the dict raises here rather than being dropped silently three
cells later.


In [ ]:
tagged = pd.read_excel(TAGGED_XLSX, sheet_name=SHEET_TAGGED, dtype=str)
tagged.columns = [norm_label(c) for c in tagged.columns]
tagged['raw_category'] = tagged['Category'].map(norm_label)

_raws = set(tagged.loc[tagged['raw_category'] != '', 'raw_category'])
_unmapped = sorted(_raws - set(RAW_TO_CANONICAL))
_stale = sorted(set(RAW_TO_CANONICAL) - _raws)
_bad_target = sorted(set(RAW_TO_CANONICAL.values()) - set(CANONICAL))
assert not _unmapped, f'raw labels with no canonical target: {_unmapped}'
assert not _stale, f'RAW_TO_CANONICAL keys that no longer appear in the data: {_stale}'
assert not _bad_target, f'mapping targets that are not canonical categories: {_bad_target}'

# An untagged row is not a mystery category - the tagger left it blank because the
# ticket does not say enough. It maps to the same place the explicit label does.
UNTAGGED = 'Not enough info'
tagged['canonical'] = tagged['raw_category'].map(RAW_TO_CANONICAL).fillna(UNTAGGED)

# ---- Re-key the tagger's own descriptions and synonyms onto the clean labels.
# Their wording is the single most useful thing in the workbook for a prompt, and
# it would be lost silently if a label were renamed on either side.
_unknown_tagger = sorted(set(TAGGER_DESC_RAW) - set(TAGGER_LABEL_TO_CANONICAL))
_bad_tagger_target = sorted(set(TAGGER_LABEL_TO_CANONICAL.values()) - set(CANONICAL))
assert not _unknown_tagger, (
    f'{SHEET_CANON} tab labels with no canonical target: {_unknown_tagger}')
assert not _bad_tagger_target, (
    f'TAGGER_LABEL_TO_CANONICAL points outside CANONICAL: {_bad_tagger_target}')
FROM_DISTINCT_TAB = set(TAGGER_LABEL_TO_CANONICAL.values())
TAGGER_DESCRIPTIONS = {TAGGER_LABEL_TO_CANONICAL[k]: v for k, v in TAGGER_DESC_RAW.items() if v}
TAGGER_SYNONYMS = {TAGGER_LABEL_TO_CANONICAL[k]: v
                   for k, v in TAGGER_SYN_RAW.items() if k in TAGGER_LABEL_TO_CANONICAL and v}
print(f"Carried over from the {SHEET_CANON} tab: {len(TAGGER_DESCRIPTIONS)} descriptions, "
      f'{len(TAGGER_SYNONYMS)} synonym lists\n')

_unreachable = [c for c in CANONICAL if c not in set(RAW_TO_CANONICAL.values())]
print(f'{len(CANONICAL)} canonical categories, {len(RAW_TO_CANONICAL)} raw labels mapped, '
      f'0 unmapped.')
print(f'Carried over from the {SHEET_CANON} tab with no rows in the tagged data yet: '
      f'{_unreachable}\n')

_counts = tagged['canonical'].value_counts()
print(f'{"canonical category":<44} {"rows":>6} {"share":>7}  raw labels folded in')
for _c in CANONICAL:
    _n = int(_counts.get(_c, 0))
    _folded = sum(1 for k, v in RAW_TO_CANONICAL.items() if v == _c)
    print(f'{_c:<44} {_n:>6} {_n / len(tagged) * 100:>6.1f}%  {_folded}')
print(f'{"TOTAL":<44} {len(tagged):>6}')
assert _counts.sum() == len(tagged), 'rows lost in the mapping'


### Write the mapping out for review


In [ ]:
_rows = []
for _c in CANONICAL:
    _members = sorted((k for k, v in RAW_TO_CANONICAL.items() if v == _c),
                      key=lambda k: -int(tagged['raw_category'].eq(k).sum()))
    for _i, _m in enumerate(_members):
        _rows.append({'canonical': _c if _i == 0 else '', 'raw_label': _m,
                      'rows': int(tagged['raw_category'].eq(_m).sum())})
    if not _members:
        _rows.append({'canonical': _c, 'raw_label': f'(none yet - from the {SHEET_CANON} tab)',
                      'rows': 0})
tax_map = pd.DataFrame(_rows)

_md = [
    '# Registration ID / BulkID run — taxonomy consolidation',
    '',
    f'How the {len(RAW_TO_CANONICAL)} raw labels in `{TAGGED_XLSX.name}` `[{SHEET_TAGGED}]` map '
    f'onto the {len(CANONICAL)} cleaned categories used by `{MOD_T}`.',
    '',
    f'Backbone: the `{SHEET_CANON}` tab, minus {len(NOT_CATEGORIES)} rows that are notes rather '
    'than categories. Additions: the categories carrying real volume in the raw labels that the '
    f'`{SHEET_CANON}` tab has no home for.',
    '',
    f'Generated by `notebooks/16_reg_id_2026_bulkid_taxonomy.ipynb`. Do not edit by hand.',
    '',
    '| canonical category | definition | raw labels folded in | rows |',
    '|---|---|---|---|',
]
for _c, _d in CANONICAL.items():
    _members = sorted((k for k, v in RAW_TO_CANONICAL.items() if v == _c),
                      key=lambda k: -int(tagged['raw_category'].eq(k).sum()))
    _n = int(tagged['canonical'].eq(_c).sum())
    _md.append(f'| **{_c}** | {_d} | {"; ".join(f"`{m}`" for m in _members) or "_(none yet)_"} '
               f'| {_n} |')
_md += ['', f'Untagged rows ({int(tagged["raw_category"].eq("").sum())}) count as '
            f'`{UNTAGGED}`.', '']
TAX_MAP_MD.write_text('\n'.join(_md), encoding='utf-8')
print(f'Wrote {TAX_MAP_MD.relative_to(BASE)} ({len(CANONICAL)} categories, {len(_md)} lines)')


## 3. Write the pipeline definition workbook

`rca build` consumes three sheets — `modules`, `fields`, `taxonomy` — and **assigns
taxonomy codes by row order**, so codes are never written by hand. The order of `CANONICAL` fixes
`RIDT1.1` … `RIDT34.1`.

Each taxonomy row's description is the canonical definition plus the tagger's own wording and
synonym list where the `Distinct` tab had one, so the prompt carries the vocabulary the tagger
actually used rather than a paraphrase of it.


In [ ]:
# ---- Shared prompt language ---------------------------------------------
G_THREAD = ('Read the whole thread before deciding. The customer opens, the agent replies, and '
            'the request is often restated or corrected further down.')
G_NEED = ('Classify on what the CUSTOMER is asking for, not on what the agent did about it. An '
          'agent who fulfils a request by issuing credentials, sending a link or raising a ticket '
          'does not change what was asked.')
G_ONE = 'Exactly one code. If two seem to fit, pick the one the customer would name themselves.'
# Added after gate 3: the model returned a null code on an agent-only note,
# reasoning there was nothing to classify. That is what the catch-all is for.
G_NEVER_BLANK = (
    'Never return an empty code. Every ticket gets one. If the thread does not say enough to '
    'classify - an agent-only note with no customer request, an auto-reply, a bare '
    'acknowledgement, an attachment with no text - that is not a reason to leave the field '
    'blank, it is the reason the catch-all category exists. Use it.')

# The trap this whole notebook is built around, stated to the model in the words
# of the measurement that found it. 356 of the 607 regex false positives are
# comp-access tickets the agent answered with registration IDs.
ADJ_RULE = (
    'You are shown the CUSTOMER side of the conversation, with the agent reply removed. A small '
    'number of tickets were logged by an agent and contain no customer turn at all - for those '
    'you are shown the whole note, and you should read it for what the customer was asking for. '
    'A phrase such as registration ID or bulk ID appearing in the text is not enough on its own. '
    'Answer '
    'YES only when the customer is asking for the IDs or credentials themselves - to be created, '
    'added, removed, exchanged, extended, counted or explained. Answer NO when the customer is '
    'asking for something else and merely mentions IDs in passing: quoting an ID they were given '
    'earlier, asking for free or trial access without specifying how it should be delivered, '
    'reporting that an ID they already hold will not work as part of a different problem, or '
    'pasting a previous email that contained one.')

ADJ_YES = 'Registration or Bulk ID request'
ADJ_NO  = 'Not a Registration or Bulk ID request'
ADJ_ORDER = [ADJ_YES, ADJ_NO]
ADJ_DEFS = {
    ADJ_YES: (
        'The customer is asking for Registration IDs, RegIDs, Bulk IDs or remote credentials '
        'themselves - to create, add, delete, exchange, extend or renew them, to be told how many '
        'remain, or to be told how they work. The IDs are the subject of the request.'),
    ADJ_NO: (
        'The customer wants something else. The matched phrase is incidental: an ID quoted as '
        'reference, a request for complimentary or trial access that does not specify '
        'credentials, a sign-in or entitlement problem that happens to involve an ID the customer '
        'already has, or quoted text from an earlier reply.'),
}

# ---- modules sheet -------------------------------------------------------
MOD_DESC_A = (
    'Decide whether this customer is actually requesting Registration IDs, Bulk IDs or remote '
    f'credentials. {G_THREAD} {G_NEED} {G_ONE} '
    f'Assign the single best-fitting LEAF code (e.g. {PREFIX_A}1.1). {ADJ_RULE} '
    f'Never return an empty code - if the text does not show the customer asking for IDs, for '
    f'any reason including that it shows nothing at all, the answer is {ADJ_NO}.')

MOD_DESC_T = (
    'Classify this customer support ticket into exactly one category from the taxonomy below. '
    'Each category carries its definition and, where the original tagger recorded them, the '
    'synonyms and phrasings they used for it. Match on what the definition describes, not on '
    f'surface keywords. {G_THREAD} {G_NEED} {G_ONE} {G_NEVER_BLANK} '
    f'The catch-all is {UNTAGGED}. '
    f'Assign the single best-fitting LEAF code (e.g. {PREFIX_T}1.1).')

modules_rows = [
    {'module_name': MOD_A, 'taxonomy_prefix': PREFIX_A, 'taxonomy_file': f'{MOD_A}.json',
     'reorder_group': 'A', 'reorder_order': 1, 'enabled': 'TRUE', 'description': MOD_DESC_A,
     'notes': ('Runs only on the tickets whose CUSTOMER turns match the Tableau '
               'Registration/Bulk ID regex. Input text is the customer turns alone, not the full '
               'thread. Contains an AUTHORED rule (ADJ_RULE) derived from measuring the regex '
               'against the manually tagged corpus - not from any specification.')},
    {'module_name': MOD_T, 'taxonomy_prefix': PREFIX_T, 'taxonomy_file': f'{MOD_T}.json',
     'reorder_group': 'B', 'reorder_order': 2, 'enabled': 'TRUE', 'description': MOD_DESC_T,
     'notes': (f'Runs on every ticket. Taxonomy consolidated from the {SHEET_CANON} tab of '
               f'{TAGGED_XLSX.name} plus the high-volume categories missing from it; see '
               f'docs/{TAX_MAP_MD.name}.')},
]

# ---- fields sheet --------------------------------------------------------
def _fields_for(stub, prefix, what, alternative_phrase):
    return [
        {'module_name': None, 'field_name': f'{stub}_code', 'field_type': 'code',
         'description': 'Assigned leaf code',
         'prompt_instruction': (
             f'Assign the single leaf-level code (e.g. {prefix}1.1) for the {what} that best '
             'matches this ticket. Read each Definition in full and match on what the ticket '
             'shows, not on surface keywords. Only use codes from the taxonomy.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'TRUE', 'notes': None},
        {'module_name': None, 'field_name': f'{stub}_justification', 'field_type': 'str',
         'description': 'Why this code was chosen over the nearest alternative',
         'prompt_instruction': (
             f'In one or two sentences, explain why this {what} applies and why '
             f'{alternative_phrase} does not. Refer to what the ticket actually shows. Do not use '
             'apostrophes or quotation marks in your answer.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
        {'module_name': None, 'field_name': f'{stub}_evidence', 'field_type': 'str',
         'description': 'Verbatim supporting quote from the transcript',
         'prompt_instruction': (
             'Copy verbatim the single sentence or phrase from the transcript that most strongly '
             'supports this choice. Do not paraphrase, infer, or add words that are not in the '
             'transcript. If no direct evidence exists, return an empty string.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
    ]


fields_rows = []
for _mod, _stub, _prefix, _what, _alt in (
        (MOD_A, STUB_A, PREFIX_A, 'verdict', 'the other verdict'),
        (MOD_T, STUB_T, PREFIX_T, 'category', 'the closest alternative category')):
    for _f in _fields_for(_stub, _prefix, _what, _alt):
        _f['module_name'] = _mod
        fields_rows.append(_f)

# ---- taxonomy sheet ------------------------------------------------------
# L1 carries the label only; the definition goes on the L2 leaf, because
# taxonomy_to_rich_prompt prints both levels.
def _leaf_description(label: str) -> str:
    """Canonical definition, plus the tagger's own wording and synonyms if any."""
    parts = [f'Definition: {CANONICAL[label]}']
    _own = TAGGER_DESCRIPTIONS.get(label, '')
    if _own and _own.lower() not in ('nan', ''):
        parts.append(f"Tagger's own note: {_own}")
    _syn = TAGGER_SYNONYMS.get(label, '')
    if _syn and _syn.lower() not in ('nan', ''):
        parts.append(f'Phrasings seen in real tickets: {_syn}')
    _folded = sorted(k for k, v in RAW_TO_CANONICAL.items() if v == label and k != label)
    if _folded:
        parts.append(f'Previously tagged as: {"; ".join(_folded)}.')
    return ' '.join(parts)


taxonomy_rows = []
for _label in ADJ_ORDER:
    taxonomy_rows.append({'module_name': MOD_A, 'level': 'L1', 'label': _label,
                          'description': None, 'notes': None})
    taxonomy_rows.append({'module_name': MOD_A, 'level': 'L2', 'label': _label,
                          'description': f'Definition: {ADJ_DEFS[_label]}',
                          'notes': 'AUTHORED for this run. See ADJ_RULE.'})
for _label in CANONICAL:
    taxonomy_rows.append({'module_name': MOD_T, 'level': 'L1', 'label': _label,
                          'description': None, 'notes': None})
    taxonomy_rows.append({'module_name': MOD_T, 'level': 'L2', 'label': _label,
                          'description': _leaf_description(_label),
                          'notes': (f'From the {SHEET_CANON} tab.'
                                    if _label in FROM_DISTINCT_TAB
                                    else 'Added: present in the raw labels, absent from the '
                                         f'{SHEET_CANON} tab.')})

# ---- write ---------------------------------------------------------------
_COLS = {
    'modules': ['module_name', 'taxonomy_prefix', 'taxonomy_file', 'reorder_group',
                'reorder_order', 'enabled', 'description', 'notes'],
    'fields': ['module_name', 'field_name', 'field_type', 'description', 'prompt_instruction',
               'in_output', 'explode_list', 'validate_enum', 'notes'],
    'taxonomy': ['module_name', 'level', 'label', 'description', 'notes'],
}
_frames = {'modules': pd.DataFrame(modules_rows)[_COLS['modules']],
           'fields': pd.DataFrame(fields_rows)[_COLS['fields']],
           'taxonomy': pd.DataFrame(taxonomy_rows)[_COLS['taxonomy']]}

with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as _xw:
    for _sheet, _df in _frames.items():
        _df.to_excel(_xw, sheet_name=_sheet, index=False)

# ---- round-trip: what rca build will actually read --------------
_back = pd.read_excel(DEFN_XLSX, sheet_name=None, dtype=str)
assert set(_back) == {'modules', 'fields', 'taxonomy'}, f'unexpected sheets: {sorted(_back)}'
assert len(_back['modules']) == 2, f'expected 2 modules, wrote {len(_back["modules"])}'
assert len(_back['fields']) == 6, f'expected 6 fields, wrote {len(_back["fields"])}'
assert len(_back['taxonomy']) == 2 * (len(ADJ_ORDER) + len(CANONICAL)), 'taxonomy row count'

# Excel silently truncates a cell over 32,767 chars.
for _s, _df in _back.items():
    for _c in _df.columns:
        _long = _df[_c].dropna().astype(str).str.len().max()
        assert pd.isna(_long) or _long < 32_000, f'{_s}.{_c} is {_long} chars - near the Excel cap'

# Every definition must have survived the write unchanged.
_tb = _back['taxonomy']
for _label, _defn in list(ADJ_DEFS.items()) + list(CANONICAL.items()):
    _row = _tb[(_tb.level == 'L2') & (_tb.label == _label)]
    assert len(_row) == 1, f'expected exactly one L2 row for {_label!r}, found {len(_row)}'
    assert _defn in _row.iloc[0]['description'], f'definition did not survive the write: {_label}'
assert ADJ_RULE in _back['modules'].set_index('module_name').loc[MOD_A, 'description'], \
    'the authored adjudication rule did not survive the write'

print(f'Wrote {DEFN_XLSX.relative_to(BASE)}')
for _s, _df in _frames.items():
    print(f'  {_s:<9} {len(_df):>3} rows')
print(f'\n{MOD_A}: {len(ADJ_ORDER)} leaves   {MOD_T}: {len(CANONICAL)} leaves')


## 4. Build the pipeline artefacts

Dry run first — it reports what would be written without touching anything.


In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--dry-run'], 'build_pipeline --dry-run')


In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', *ALL_MODULES], 'build_pipeline')

# ---- Verify the built shape ---------------------------------------------
cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
assert set(cfg['modules']) == set(ALL_MODULES), (
    f'config modules mismatch\n  missing: {set(ALL_MODULES) - set(cfg["modules"])}'
    f'\n  extra  : {set(cfg["modules"]) - set(ALL_MODULES)}')

for _module, _want in ((MOD_A, len(ADJ_ORDER)), (MOD_T, len(CANONICAL))):
    assert (TAXONOMY_DIR / f'{_module}.json').exists(), f'missing taxonomy for {_module}'
    assert (MODULES_DIR / f'{_module}.py').exists(), f'missing {_module}.py'
    _got = len(leaf_labels(_module))
    assert _got == _want, f'{_module}: expected {_want} leaves, got {_got}'
    assert cfg['modules'][_module]['taxonomy_prefix'] == MODULE_PREFIXES[_module]

# Field stubs must not collide with the prefixes drop_enrichment_cols() strips,
# or the results would vanish from the output on a re-run.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
for _stub in (STUB_A, STUB_T):
    for _bad in _STRUCTURAL_PREFIXES:
        assert not f'{_stub}_'.startswith(_bad), f'field stub {_stub}_ collides with {_bad!r}'
# Same trap for the columns this notebook adds itself in sections 6 and 11.
OWN_COLS = ['rid_regex_flag', 'rid_regex_flag_fulltext', 'rid_regex_term', 'rid_customer_chars',
            'rid_no_customer_turn', 'rid_adjudication', 'rid_category', 'rid_category_source',
            'rid_justification', 'rid_evidence']
for _c in OWN_COLS:
    for _bad in _STRUCTURAL_PREFIXES:
        assert not _c.startswith(_bad), f'output column {_c!r} collides with {_bad!r}'

# Prefixes are parsed with a greedy ^[A-Z]+ elsewhere in the pipeline, so
# neither may be a prefix of the other.
assert not PREFIX_A.startswith(PREFIX_T) and not PREFIX_T.startswith(PREFIX_A), \
    f'{PREFIX_A} and {PREFIX_T} overlap'

ADJ_LABELS = leaf_labels(MOD_A)                          # code -> label
TAX_LABELS = leaf_labels(MOD_T)
CODE_OF_ADJ = {v: k for k, v in ADJ_LABELS.items()}
CODE_OF_TAX = {v: k for k, v in TAX_LABELS.items()}
assert len(CODE_OF_TAX) == len(TAX_LABELS), 'duplicate canonical labels'
assert set(TAX_LABELS.values()) == set(CANONICAL), 'built labels differ from CANONICAL'
YES_CODE = CODE_OF_ADJ[ADJ_YES]
BULKID_LABEL = 'BulkIDs / Registration IDs'
assert BULKID_LABEL in CODE_OF_TAX, 'the BulkIDs category is not in the built taxonomy'

print(f'\nBuild verified: {len(ALL_MODULES)} modules, '
      f'{len(ADJ_LABELS) + len(TAX_LABELS)} leaves total.\n')
print(f'{MOD_A}:')
for _code, _label in ADJ_LABELS.items():
    print(f'  {_code:<9} {_label}')
print(f'\n{MOD_T}:')
for _code, _label in TAX_LABELS.items():
    print(f'  {_code:<9} {_label}')


## 5. Gate 1 — review the exact prompts *(free, read this output)*

Renders both system prompts exactly as the model will receive them, then asserts that every
canonical definition and the authored adjudication rule survive into the prompt **verbatim**. A
dropped definition or a mis-wired prefix is invisible in the output of a run and expensive to find
afterwards.


In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
prompts = {}
for module in ALL_MODULES:
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    name = f'generated.modules.{module}'
    mod = importlib.reload(sys.modules[name]) if name in sys.modules \
        else importlib.import_module(name)
    prompts[module] = mod._build_system_prompt(
        taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))

# ---- Verbatim check -----------------------------------------------------
checked = 0
for _label, _defn in CANONICAL.items():
    assert _label in prompts[MOD_T], f'category missing from the prompt: {_label!r}'
    assert _defn in prompts[MOD_T], f'definition missing for {_label}: {_defn[:70]!r}'
    checked += 2
for _label, _defn in ADJ_DEFS.items():
    assert _label in prompts[MOD_A], f'verdict missing from the prompt: {_label!r}'
    assert _defn in prompts[MOD_A], f'verdict definition missing: {_label}'
    checked += 2
for _text, _module, _what in ((ADJ_RULE, MOD_A, 'authored adjudication rule'),
                              (G_NEVER_BLANK, MOD_T, 'never-blank instruction'),
                              (G_THREAD, MOD_A, 'full-thread instruction'),
                              (G_THREAD, MOD_T, 'full-thread instruction'),
                              (G_NEED, MOD_A, 'customer-not-agent instruction'),
                              (G_NEED, MOD_T, 'customer-not-agent instruction')):
    assert _text in prompts[_module], f'{_what} missing from {_module}'
    checked += 1
print(f'Verbatim check: all {checked} strings present in the prompts as written.\n')

# ---- Per-module fingerprints -------------------------------------------
# A prompt is not the only thing that changes the answer. Sampling settings do
# too, and so does batch composition - two records in one request see each
# other's context. Fold them in, or editing temperature would silently reuse
# answers sampled at the old one.
_tuning_sig = '|'.join(f'{k}={TUNING[k]}' for k in OUTPUT_AFFECTING)
# The regex and the speaker split decide which rows reach MOD_A and what text
# they carry, so they belong in its fingerprint too.
_preproc = f'{SHORT_CHAR_THRESHOLD}|{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}'
PROMPT_FINGERPRINTS = {}
for _m in ALL_MODULES:
    _extra = _tuning_sig + '|' + _preproc + ('|' + REGID_PATTERN if _m == MOD_A else '')
    PROMPT_FINGERPRINTS[_m] = hashlib.sha256(
        (prompts[_m] + _extra).encode('utf-8')).hexdigest()[:16]
print(f'Output-affecting tuning folded in: {_tuning_sig}\n')

print(f'{"module":<18} {"chars":>7} {"~tokens":>8}  fingerprint')
for module in ALL_MODULES:
    p = prompts[module]
    print(f'{module:<18} {len(p):>7,} {len(p) // 4:>8,}  {PROMPT_FINGERPRINTS[module]}')


In [ ]:
print('=' * 78)
print(f'FULL ADJUDICATION SYSTEM PROMPT  ({MOD_A})')
print('=' * 78)
print(prompts[MOD_A])


In [ ]:
print('=' * 78)
print(f'FULL TAXONOMY SYSTEM PROMPT  ({MOD_T})')
print('=' * 78)
print(prompts[MOD_T])


## 6. Gate 2 — preprocess the tickets *(free)*

Four transformations, each asserted:

**Split the speakers.** `customer_text` is the customer's turns alone — about 62% of the thread by
character. This is what the regex and the adjudicator see; the taxonomy module gets the full
thread, because the agent's reply is often the clearest statement of what the ticket was about.
Every conversation carries markers, but ~5.7% contain no customer turn at all (an agent-logged
note, a phone summary); those fall back to the full text rather than dropping out of the regex
population.

**Apply the regex, twice.** Once on customer turns (the population that goes to the adjudicator)
and once on the full conversation, kept as an audit column so the Tableau figure stays
reconcilable.

**Cap the conversations.** Anything over 16,000 characters is sent as head 4,000 + tail 11,900 with
a visible marker. The source export is itself already cut at 32,767 characters by Excel, so some
conversations are incomplete before this notebook sees them.

**Rule-assign the empty ones.** A ticket with under 40 characters of payload cannot evidence
anything; it is set to `Not enough info` by rule with no LLM call, and flagged as such in
`rid_category_source`.


In [ ]:
_t0 = time.time()
raw = pd.read_excel(TICKETS_XLSX, dtype=str)
raw.columns = [norm_label(c) for c in raw.columns]
print(f'{TICKETS_XLSX.name}: {len(raw):,} rows x {len(raw.columns)} cols '
      f'({time.time() - _t0:.0f}s)')
print(f'  columns: {list(raw.columns)}')

assert ID_COL in raw.columns and TEXT_COL in raw.columns, f'expected {ID_COL} and {TEXT_COL}'
raw[ID_COL] = raw[ID_COL].astype(str).str.strip()
assert raw[ID_COL].is_unique, f'{ID_COL} is not unique - the merge in section 11 would fan out'

tickets = raw.copy()
tickets[TEXT_COL] = tickets[TEXT_COL].fillna('')
tickets['parsed_date'] = pd.to_datetime(tickets[DATE_COL], errors='coerce')
tickets['month'] = tickets['parsed_date'].dt.to_period('M').astype(str)
print(f'\nDate range: {tickets["parsed_date"].min()} to {tickets["parsed_date"].max()} '
      f'({tickets["month"].nunique()} months)')
print(f'Channels  : {tickets["Channel"].value_counts().to_dict()}')

# ---- 1. Speaker split ---------------------------------------------------
tickets['customer_text'] = tickets[TEXT_COL].map(customer_turns)
tickets['rid_no_customer_turn'] = ~tickets[TEXT_COL].str.contains(_CUSTOMER_RE)

_has_marker = tickets[TEXT_COL].str.contains(_SPEAKER_ANY)
print(f'\nConversations carrying any Customer:/Agent: marker: {_has_marker.sum():,} '
      f'({_has_marker.mean() * 100:.1f}%)')
print(f'  with no CUSTOMER turn, so falling back to the full text: '
      f'{int(tickets["rid_no_customer_turn"].sum()):,} '
      f'({tickets["rid_no_customer_turn"].mean() * 100:.1f}%)')
_cust_share = (tickets['customer_text'].str.len().sum()
               / max(tickets[TEXT_COL].str.len().sum(), 1))
print(f'customer_text is {_cust_share:.0%} of the conversation text by character')

# The fallback is a deliberate choice, not an accident. If it ever covers most
# of the file then the speaker split has broken on this export and the whole
# precision argument for running the regex on customer text is void.
assert tickets['rid_no_customer_turn'].mean() < 0.25, (
    'over a quarter of conversations have no customer turn - the speaker split is not '
    'working on this export, and running the regex on customer text would be meaningless')


In [ ]:
# ---- 2. The regex, on both populations ----------------------------------
tickets['rid_regex_flag'] = tickets['customer_text'].str.lower().map(
    lambda t: bool(REGID_RX.search(t)))
tickets['rid_regex_flag_fulltext'] = tickets[TEXT_COL].str.lower().map(
    lambda t: bool(REGID_RX.search(t)))
tickets['rid_regex_term'] = tickets['customer_text'].map(regid_match)
tickets['rid_customer_chars'] = tickets['customer_text'].str.len()

_n_cust = int(tickets['rid_regex_flag'].sum())
_n_full = int(tickets['rid_regex_flag_fulltext'].sum())
print(f'Regex on the customer turns   : {_n_cust:>6,}  ({_n_cust / len(tickets) * 100:.1f}%)')
print(f'Regex on the whole thread     : {_n_full:>6,}  ({_n_full / len(tickets) * 100:.1f}%)')
print(f'  matched only in the agent reply: {_n_full - _n_cust:,} - these are the comp-access '
      f'shape the precision measurement found, and they do not go to the adjudicator')
assert _n_cust <= _n_full, 'customer text is a subset of the thread; this cannot happen'
assert _n_cust > 0, 'the regex matched nothing - check the speaker split'

print(f'\nMost common matched terms:')
print(tickets.loc[tickets['rid_regex_flag'], 'rid_regex_term'].value_counts().head(15).to_string())

print('\n--- 10 matched snippets, in context ---')
for _, _r in tickets[tickets['rid_regex_flag']].head(10).iterrows():
    _t = _r['customer_text'].lower()
    _i = REGID_RX.search(_t).start()
    print(f'  [{_r[ID_COL]}] ...{_t[max(0, _i - 90):_i + 90]}...'.replace('\n', ' '))


In [ ]:
# ---- 3. Truncation ------------------------------------------------------
TRUNC_MARKER = '\n\n...[MIDDLE OF CONVERSATION OMITTED FOR LENGTH]...\n\n'
assert TRUNC_HEAD + len(TRUNC_MARKER) + TRUNC_TAIL <= TRUNC_CAP, 'head+marker+tail exceeds the cap'


def cap(text: str) -> str:
    text = str(text)
    if len(text) <= TRUNC_CAP:
        return text
    return text[:TRUNC_HEAD] + TRUNC_MARKER + text[-TRUNC_TAIL:]


tickets['conversation_chars_original'] = tickets[TEXT_COL].str.len()
tickets['conversation_sent'] = tickets[TEXT_COL].map(cap)
tickets['customer_sent'] = tickets['customer_text'].map(cap)
tickets['was_truncated'] = tickets['conversation_chars_original'] > TRUNC_CAP
assert tickets['conversation_sent'].str.len().max() <= TRUNC_CAP, 'cap() let something through'

print(f'Truncated to {TRUNC_CAP:,} chars : {int(tickets["was_truncated"].sum()):,} conversations')
print(f'Already cut at 32,767 by the source export: '
      f'{int((tickets["conversation_chars_original"] >= 32_767).sum()):,} - incomplete at source')

# ---- 4. Short-ticket rule ------------------------------------------------
def payload_len(text) -> int:
    """Conversation length with the speaker markers themselves discounted."""
    return len(_SPEAKER_ANY.sub('', str(text)).strip())


tickets['payload_chars'] = tickets[TEXT_COL].map(payload_len)
tickets['too_short'] = tickets['payload_chars'] < SHORT_CHAR_THRESHOLD
_short = tickets[tickets['too_short']]
print(f'\nUnder {SHORT_CHAR_THRESHOLD} payload chars: {len(_short):,} tickets '
      f'-> rule-assigned "{UNTAGGED}", no LLM call')
if len(_short):
    print('  examples: ' + ' | '.join(repr(t[:60]) for t in _short[TEXT_COL].head(3)))
# A short ticket that nevertheless matched the regex still goes to the
# adjudicator - "need 5 more reg ids" is 19 characters and perfectly classifiable.
_short_flagged = int((tickets['too_short'] & tickets['rid_regex_flag']).sum())
print(f'  of which {_short_flagged} matched the regex and still go to {MOD_A}')


In [ ]:
# ---- Write the two inputs -----------------------------------------------
to_classify = tickets[~tickets['too_short']].copy()
to_adjudicate = tickets[tickets['rid_regex_flag']].copy()

to_classify[[ID_COL, 'conversation_sent']].rename(
    columns={'conversation_sent': TEXT_COL}).to_csv(INPUT_T, index=False, encoding='utf-8')
to_adjudicate[[ID_COL, 'customer_sent']].rename(
    columns={'customer_sent': TEXT_COL}).to_csv(INPUT_A, index=False, encoding='utf-8')

# The adjudicator must not be able to see the agent's reply - that is the entire
# reason precision improves. Prove it on the file that will actually be read,
# not on the frame in memory.
#
# The exception is the fallback population: a ticket with no customer turn at all
# (an agent-logged note, a phone summary) is sent whole, because the alternative
# is sending nothing. Those rows DO carry the agent's words, they are flagged in
# rid_no_customer_turn, and the adjudicator prompt says so. Every other row must
# be clean, and that is what is asserted here.
_check = pd.read_csv(INPUT_A, dtype=str).merge(
    tickets[[ID_COL, 'rid_no_customer_turn']], on=ID_COL, how='left', validate='1:1')
assert len(_check) == len(to_adjudicate), 'adjudicator input lost rows on write'
assert _check['rid_no_customer_turn'].notna().all(), 'an adjudicator row is not in tickets'

_marked = _check[TEXT_COL].fillna('').str.contains(_SPEAKER_ANY)
_leaked = int((_marked & ~_check['rid_no_customer_turn']).sum())
assert _leaked == 0, (
    f'{_leaked} adjudicator rows have a customer turn AND still carry a speaker marker - '
    f'the split dropped text it should have kept apart')
_fallback = int(_check['rid_no_customer_turn'].sum())
print(f'Adjudicator input: {len(_check) - _fallback:,} rows are customer turns only, '
      f'{_fallback:,} are whole agent-logged notes with no customer turn '
      f'({_fallback / max(len(_check), 1) * 100:.1f}%)')
assert _fallback / max(len(_check), 1) < 0.5, (
    'most of the adjudicator population is fallback text - the precision argument does not hold '
    'and the customer-turns-only framing in the prompt would be misleading')
assert set(to_adjudicate[ID_COL]) <= set(tickets[ID_COL])

print(f'{INPUT_T.relative_to(BASE)}  {len(to_classify):>6,} rows  (full thread, every ticket '
      f'with a payload)')
print(f'{INPUT_A.relative_to(BASE)}  {len(to_adjudicate):>6,} rows  (customer turns, regex hits '
      f'only)')
print(f'\nAccounting: {len(to_classify):,} to {MOD_T} + {int(tickets["too_short"].sum()):,} '
      f'rule-assigned = {len(tickets):,}')
assert len(to_classify) + int(tickets['too_short'].sum()) == len(tickets), 'rows unaccounted for'


## 7. Generate the profiles and arm the fingerprint guard

`rca run` applies a profile key only when the matching CLI arg is still at its parser
default, via a `hasattr` guard — so **a key it does not recognise is silently ignored**. Writing
`max_char` for `max_chars` would not error, it would just run at the default. The profile writer
below checks every key against `--help` before writing.


In [ ]:
# ---- Which keys does rca run actually accept? -------------------
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests, \
    f'could not parse rca run flags (got {sorted(_dests)})'


def write_profile(path: Path, *, comment, input_rel, output_rel, modules) -> Path:
    spec = {'_comment': comment, 'input': input_rel, 'output': output_rel,
            'taxonomy_dir': 'generated/taxonomies',
            'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': modules, **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


write_profile(
    PROFILE_A,
    comment=('Adjudicates the tickets whose CUSTOMER turns match the Tableau Registration/Bulk ID '
             'regex: is the customer really asking for the IDs, or did the phrase arrive some '
             'other way? The Conversation column of this input holds the CUSTOMER TURNS ONLY, '
             'with the agent reply deliberately withheld, because measuring the regex against '
             'the manually tagged corpus showed 356 of 607 false positives were '
             'complimentary-access tickets the agent answered by issuing registration IDs. The '
             'exception is tickets that contain no customer turn at all (agent-logged notes, '
             'phone summaries), which are sent whole - about 11% of this input, flagged in the '
             'deliverable as rid_no_customer_turn. Input is written by '
             f'notebooks/16_reg_id_2026_bulkid_taxonomy.ipynb, not the raw file.'),
    input_rel=INPUT_A.relative_to(BASE).as_posix(),
    output_rel=(OUT_DIR / 'reg_id_2026_adjudicate.xlsx').relative_to(BASE).as_posix(),
    modules=[MOD_A])

write_profile(
    PROFILE_T,
    comment=('Assigns one cleaned-taxonomy category to every ticket with a payload. Taxonomy '
             f'consolidated from the {SHEET_CANON} tab of {TAGGED_XLSX.name} plus the '
             f'high-volume categories missing from it - see docs/{TAX_MAP_MD.name}. Input is '
             f'the full conversation, capped at {TRUNC_CAP} chars, written by '
             f'notebooks/16_reg_id_2026_bulkid_taxonomy.ipynb.'),
    input_rel=INPUT_T.relative_to(BASE).as_posix(),
    output_rel=(OUT_DIR / 'reg_id_2026_taxonomy.xlsx').relative_to(BASE).as_posix(),
    modules=[MOD_T])

for _p in (PROFILE_A, PROFILE_T):
    print(f'--- {_p.relative_to(BASE)}')
    print(_p.read_text(encoding='utf-8'))


In [ ]:
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list:
    """The checkpoint artefacts rca run keeps for a given input file."""
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    """Read a {module: hash} fingerprint file, tolerating older formats."""
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, module: str) -> None:
    """Drop this input's checkpoint if it was produced by a different prompt.

    The pipeline resumes from runs/checkpoints/<stem>_checkpoint.jsonl whenever
    every module already has a result for a record - it logs `skip=N todo=0` and
    makes no LLM calls. That is what you want after an interruption and exactly
    what you do not want after editing the taxonomy.
    """
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    if stored.get(module) == want:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  {module}: prompt changed since that checkpoint '
              f'({stored.get(module) or "no fingerprint"} -> {want}); '
              f'discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, module: str) -> None:
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    stored[module] = want
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, module: str, input_path=None, output_path=None,
              label: str = '', fresh: bool = False) -> Path:
    """Run one pipeline stage as a subprocess and return its output path.

    A subprocess rather than an in-process call so a dropped kernel does not kill
    a long run, and so MODULE_REGISTRY is loaded fresh from the current config.

    fresh=True deletes the checkpoint first, forcing real LLM calls. Use it for
    the smoke test, whose whole purpose is to exercise the path: a smoke test
    that replays cache proves nothing. Never use it for the full runs.
    """
    spec = json.loads(profile.read_text(encoding='utf-8'))
    in_path = Path(input_path) if input_path else BASE / spec['input']
    out_path = Path(output_path) if output_path else BASE / spec['output']
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if fresh:
        gone = [p.name for p in checkpoint_files(in_path) if p.exists()]
        for p in checkpoint_files(in_path):
            p.unlink(missing_ok=True)
        _fingerprint_file(in_path).unlink(missing_ok=True)
        if gone:
            print(f'  fresh=True: discarded {", ".join(gone)} so this call really runs')
    else:
        _sync_checkpoint(in_path, module)
    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run',
           '--profile', str(profile),
           '--input', str(in_path), '--output', str(out_path),
           '--modules', module]
    print(f'\n{"=" * 78}\n{label or module}\n{"=" * 78}')
    print(f'$ {" ".join(cmd)}\n')
    _t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding='utf-8', errors='replace',
                            bufsize=1, env=SUBPROCESS_ENV)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{label or module} failed (exit {rc})')
    assert out_path.exists(), f'{label or module} reported success but wrote no output'
    _stamp_checkpoint(in_path, module)
    print(f'\n{label or module}: {time.time() - _t0:.0f}s -> {out_path.relative_to(BASE)}')
    return out_path


def read_labels(path, stub: str, code_to_label: dict, out_col: str) -> pd.DataFrame:
    """Stage output -> id, label, justification, evidence. Unknown codes become ''."""
    df = read_stage_output(path)
    code = df[f'{stub}_code'].fillna('').astype(str).str.strip()
    out = pd.DataFrame({
        ID_COL: df[ID_COL],
        f'{out_col}_code': code,
        out_col: code.map(code_to_label).fillna(''),
        f'{out_col}_justification': df.get(f'{stub}_justification', '').fillna(''),
        f'{out_col}_evidence': df.get(f'{stub}_evidence', '').fillna(''),
    })
    grounded = f'{stub}_evidence_grounded'
    if grounded in df.columns:
        out[f'{out_col}_evidence_grounded'] = df[grounded]
    return out


print('Stage runner and checkpoint guard defined.')
print(f'Fingerprints armed: {PROMPT_FINGERPRINTS}')


## 8. Gate 3 — smoke test *(~$0.30)*

30 tickets through the taxonomy module and every adjudicator population that matters, with real
LLM calls (`fresh=True` — a smoke test that replays a checkpoint proves nothing).

The taxonomy sample is stratified over `Channel`: a phone note and a web form read nothing like
each other, and 30 web forms would prove only that web forms parse. `.sample()` rather than
`.head()` because the file is date-ordered and a contiguous slice over-represents whatever request
type happened to be arriving that hour.


In [ ]:
SMOKE_N = 30
SMOKE_PER_CHANNEL = 3
SMOKE_SEED = 0

_strata = to_classify['Channel'].fillna('(none)')
_top = _strata.value_counts().head(6).index.tolist()
_parts = [to_classify[_strata == s].sample(
              n=min(SMOKE_PER_CHANNEL, int((_strata == s).sum())), random_state=SMOKE_SEED)
          for s in _top]
_seeded = pd.concat(_parts)
_rest = (to_classify[~to_classify[ID_COL].isin(set(_seeded[ID_COL]))]
         .sample(n=SMOKE_N - len(_seeded), random_state=SMOKE_SEED))
smoke_tickets = pd.concat([_seeded, _rest]).reset_index(drop=True)
assert smoke_tickets[ID_COL].is_unique and len(smoke_tickets) == SMOKE_N

smoke_in_t = SMOKE_DIR / 'regid_smoke_taxonomy_input.csv'
smoke_tickets[[ID_COL, 'conversation_sent']].rename(
    columns={'conversation_sent': TEXT_COL}).to_csv(smoke_in_t, index=False, encoding='utf-8')

_months = smoke_tickets['parsed_date'].dt.to_period('M').value_counts().sort_index()
print(f'{len(smoke_tickets)} tickets across {smoke_tickets["Channel"].nunique()} channels, '
      f'{len(_months)} of {tickets["month"].nunique()} months')
print(smoke_tickets['Channel'].value_counts().to_string())
assert len(_months) >= 4, (
    'the smoke sample spans fewer than 4 months - it is clustered, and a gate on a clustered '
    'sample tells you about one week rather than the population')

smoke_t = read_labels(
    run_stage(PROFILE_T, module=MOD_T, input_path=smoke_in_t,
              output_path=SMOKE_DIR / 'smoke_taxonomy.xlsx',
              label=f'SMOKE {MOD_T} - {len(smoke_tickets)} tickets', fresh=True),
    STUB_T, TAX_LABELS, 'rid_category')

_blank_mask = smoke_t['rid_category'] == ''
_blank = int(_blank_mask.sum())
_bad = sorted(set(smoke_t.loc[_blank_mask, 'rid_category_code']) - {''})
print(f'\nBlank or off-taxonomy codes: {_blank}'
      + (f'  unknown: {_bad}' if _bad else ''))
for _, _r in smoke_t[_blank_mask].iterrows():
    print(f'  [{_r[ID_COL]}] code={_r["rid_category_code"]!r}  '
          f'{_r["rid_category_justification"]}')
assert _blank == 0, (
    'the model left a code blank or returned one outside the taxonomy. Read the justifications '
    f'above: if it is declining to classify an empty ticket, {UNTAGGED} is the answer and '
    'G_NEVER_BLANK needs to say so more plainly. Fix before spending more.')
print(smoke_t['rid_category'].value_counts().to_string())


### The adjudicator, on the population it exists for

Not a random 30. The comparison that matters is between tickets the regex flagged on the customer
side (should mostly be YES) and the 587 it flagged **only** in the agent's reply (invisible to the
adjudicator — they are not in its input at all). So this samples the flagged population and reads
the verdicts by hand.

Watch the `rid_no_customer_turn` rows in particular: those are the agent-logged notes sent whole,
the one population where the adjudicator can still see agent wording.


In [ ]:
SMOKE_ADJ_N = 30
smoke_adj = to_adjudicate.sample(n=min(SMOKE_ADJ_N, len(to_adjudicate)),
                                 random_state=SMOKE_SEED).reset_index(drop=True)
smoke_in_a = SMOKE_DIR / 'regid_smoke_adjudicate_input.csv'
smoke_adj[[ID_COL, 'customer_sent']].rename(
    columns={'customer_sent': TEXT_COL}).to_csv(smoke_in_a, index=False, encoding='utf-8')

smoke_a = read_labels(
    run_stage(PROFILE_A, module=MOD_A, input_path=smoke_in_a,
              output_path=SMOKE_DIR / 'smoke_adjudicate.xlsx',
              label=f'SMOKE {MOD_A} - {len(smoke_adj)} tickets', fresh=True),
    STUB_A, ADJ_LABELS, 'rid_adjudication')

assert int((smoke_a['rid_adjudication'] == '').sum()) == 0, 'adjudicator returned an unknown code'
print('\n' + smoke_a['rid_adjudication'].value_counts().to_string())

# Read these. The justification should reason from what the customer wrote, not
# restate the label, and a NO should name what the customer actually wanted.
_j = smoke_a.merge(
    smoke_adj[[ID_COL, 'rid_regex_term', 'customer_text', 'rid_no_customer_turn']],
    on=ID_COL)
print(f'  of which {int(_j["rid_no_customer_turn"].sum())} are agent-logged notes sent whole')
for _, _r in _j.head(10).iterrows():
    _tag = '  [no customer turn - whole note]' if _r['rid_no_customer_turn'] else ''
    print(f'\n[{_r[ID_COL]}]  matched {_r["rid_regex_term"]!r}{_tag}')
    print(f'  -> {_r["rid_adjudication"]}')
    print(f'     {_r["rid_adjudication_justification"]}')
    print(f'     evidence: {str(_r["rid_adjudication_evidence"])[:160]}')


### Decision point

Everything past here costs money. Continue when:

- the taxonomy justifications **reason from the transcript** rather than restating the label;
- no category is swallowing obviously unrelated tickets (watch `Other account settings` and
  `General information request`, the two catch-alls);
- the adjudicator's NO verdicts name what the customer actually wanted, and its YES verdicts point
  at an explicit ask for IDs rather than at a mention of one.

If a category needs rewording, edit `CANONICAL` and re-run from section 3. The fingerprint guard
will discard the affected checkpoints on the next run, so nothing stale is reused.


In [ ]:
STOP_AFTER_SMOKE = False   # set False after reading the smoke output above

if STOP_AFTER_SMOKE:
    raise SystemExit('STOP_AFTER_SMOKE is True - read the smoke output, then set it to False.')
print('Continuing to gate 4.')


## 9. Gate 4 — score the taxonomy against the manually tagged corpus *(~$3)*

A stratified 400-row sample of the 7,082 hand-tagged rows, mapped through `RAW_TO_CANONICAL`, run
through `regid_taxonomy` and scored.

**Read this score with the caveat attached.** The tagged corpus's `Document` column is mostly the
**agent reply alone** — only 1.8% of its rows carry a `Customer:` marker, against 94% of the
production data. It is a different text shape, so this measures whether the cleaned categories are
separable and whether the definitions read the way the tagger meant them. It is not the accuracy of
the production run, and nothing in the deliverable quotes it as such.

The sample is stratified on the canonical label so the small categories are represented rather than
swamped by `IP management`.


In [ ]:
GOLD_N = 400
GOLD_SEED = 0
GOLD_MIN_ROWS = 5     # categories rarer than this cannot be scored; reported, not sampled

gold_pool = tagged[tagged['Document'].fillna('').str.len() >= SHORT_CHAR_THRESHOLD].copy()
gold_pool['Document Id'] = gold_pool['Document Id'].astype(str).str.strip()
_vc = gold_pool['canonical'].value_counts()
_scorable = _vc[_vc >= GOLD_MIN_ROWS].index
print(f'Tagged rows with a usable body: {len(gold_pool):,} of {len(tagged):,}')
print(f'Categories with >= {GOLD_MIN_ROWS} rows: {len(_scorable)} of {len(CANONICAL)} '
      f'- the rest cannot be scored at this sample size:')
print(f'  {sorted(set(CANONICAL) - set(_scorable))}')

# Proportional-with-a-floor: every scorable category gets at least GOLD_MIN_ROWS
# rows, the remainder is shared out by volume. Straight proportional sampling
# would give the rare categories 1-2 rows each and an F1 of 0 or 1.
_per = (_vc[_scorable] / _vc[_scorable].sum() * GOLD_N).clip(lower=GOLD_MIN_ROWS).round().astype(int)
gold = pd.concat([gold_pool[gold_pool['canonical'] == c].sample(
                      n=min(int(_per[c]), int(_vc[c])), random_state=GOLD_SEED)
                  for c in _scorable]).reset_index(drop=True)

# The profile's id_col is INCIDENT_REF, so the gold ids wear that name here.
gold_in = SMOKE_DIR / 'regid_gold_input.csv'
gold[[ 'Document Id', 'Document']].rename(
    columns={'Document Id': ID_COL, 'Document': TEXT_COL}).assign(
    **{TEXT_COL: lambda d: d[TEXT_COL].map(cap)}).to_csv(gold_in, index=False, encoding='utf-8')
print(f'\nGold sample: {len(gold)} rows over {gold["canonical"].nunique()} categories '
      f'-> {gold_in.relative_to(BASE)}')
print(gold['canonical'].value_counts().to_string())


In [ ]:
from sklearn.metrics import classification_report, f1_score

gold_pred = read_labels(
    run_stage(PROFILE_T, module=MOD_T, input_path=gold_in,
              output_path=SMOKE_DIR / 'gold_taxonomy.xlsx',
              label=f'GOLD {MOD_T} - {len(gold)} hand-tagged rows'),
    STUB_T, TAX_LABELS, 'rid_category')

scored = gold[['Document Id', 'canonical']].rename(
    columns={'Document Id': ID_COL, 'canonical': 'truth'}).merge(
    gold_pred[[ID_COL, 'rid_category']], on=ID_COL, how='left', validate='1:1')
assert scored['rid_category'].notna().all(), 'some gold rows came back without a prediction'
print(f'Scored {len(scored)} rows\n')

print(classification_report(scored['truth'], scored['rid_category'], zero_division=0))
print(f'macro-F1    : {f1_score(scored["truth"], scored["rid_category"], average="macro", zero_division=0):.3f}')
print(f'weighted-F1 : {f1_score(scored["truth"], scored["rid_category"], average="weighted", zero_division=0):.3f}')
print(f'exact match : {(scored["truth"] == scored["rid_category"]).mean():.1%}')

# The BulkIDs class is the one this notebook is named after. 31% precision is the
# regex-on-full-text baseline it has to beat.
_b = scored['rid_category'] == BULKID_LABEL
_t = scored['truth'] == BULKID_LABEL
print(f'\n{BULKID_LABEL}: predicted {int(_b.sum())}, true {int(_t.sum())}, '
      f'precision {(_b & _t).sum() / max(_b.sum(), 1):.1%}, '
      f'recall {(_b & _t).sum() / max(_t.sum(), 1):.1%}   '
      f'(regex-on-full-text baseline: 31% precision, 97% recall)')

print('\n--- where the disagreements go (truth -> predicted, top 20) ---')
_dis = scored[scored['truth'] != scored['rid_category']]
print(_dis.groupby(['truth', 'rid_category']).size().sort_values(ascending=False)
      .head(20).to_string())


### Decision point

The full run is roughly 11,400 tickets. Continue when the confusion above is between categories
that a human would also argue about (`Add affiliation` vs `Not registered or signed in`,
`Domain management` vs `SSO / Shibboleth configuration`) rather than between unrelated ones, and
when the `BulkIDs / Registration IDs` precision clears the 31% regex baseline by a wide margin.

A category the model never predicts is a definition problem, not a model problem — reword it in
`CANONICAL` and re-run from section 3.


In [ ]:
STOP_AFTER_GOLD = False   # set False after reading the scores above

if STOP_AFTER_GOLD:
    raise SystemExit('STOP_AFTER_GOLD is True - read the scores, then set it to False.')
print('Continuing to the full run.')


## 10. The full runs

Both run as subprocesses, so a dropped kernel does not kill them. If either is interrupted,
**re-run its cell**: the checkpoint resumes and only unfinished records are paid for again. The
fingerprint guard makes that resume safe — if the prompt has changed since the checkpoint was
written, it is discarded rather than replayed.


In [ ]:
adj_out = run_stage(PROFILE_A, module=MOD_A,
                    label=f'FULL {MOD_A} - {len(to_adjudicate):,} regex-flagged tickets')
adj = read_labels(adj_out, STUB_A, ADJ_LABELS, 'rid_adjudication')
assert set(adj[ID_COL]) == set(to_adjudicate[ID_COL].astype(str)), \
    'the adjudicator returned a different id set than it was given'
print('\n' + adj['rid_adjudication'].value_counts().to_string())


In [ ]:
tax_out = run_stage(PROFILE_T, module=MOD_T,
                    label=f'FULL {MOD_T} - {len(to_classify):,} tickets')
tax = read_labels(tax_out, STUB_T, TAX_LABELS, 'rid_category')
assert set(tax[ID_COL]) == set(to_classify[ID_COL].astype(str)), \
    'the taxonomy module returned a different id set than it was given'
_blank = int((tax['rid_category'] == '').sum())
print(f'\nBlank or off-taxonomy codes: {_blank} of {len(tax):,} '
      f'({_blank / len(tax) * 100:.2f}%)')
print(tax['rid_category'].value_counts().to_string())


## 11. Merge, and settle the final label

Three sources, in precedence order:

1. **`rule`** — payload under 40 characters, set to `Not enough info` without an LLM call.
2. **`adjudicated`** — the customer's turns matched the regex *and* the adjudicator confirmed the
   customer is asking for the IDs. Final label `BulkIDs / Registration IDs`.
3. **`llm`** — the taxonomy module's answer.

A ticket the taxonomy module calls `BulkIDs / Registration IDs` without the regex ever firing keeps
that label. It is the recall backstop for phrasings the regex has no pattern for, and it is counted
separately in the funnel below.


In [ ]:
final = tickets.merge(tax, on=ID_COL, how='left', validate='1:1') \
               .merge(adj, on=ID_COL, how='left', validate='1:1')
assert len(final) == len(tickets), 'the merge changed the row count'

_adjudicated_yes = final['rid_adjudication'].eq(ADJ_YES)
final['rid_category'] = final['rid_category'].fillna('')
final['rid_adjudication'] = final['rid_adjudication'].fillna('')

final['rid_category_source'] = 'llm'
final.loc[final['too_short'], ['rid_category', 'rid_category_source']] = [UNTAGGED, 'rule']
final.loc[_adjudicated_yes, ['rid_category', 'rid_category_source']] = [BULKID_LABEL, 'adjudicated']

# A blank here would be an LLM row whose code fell outside the taxonomy. Rather
# than shipping an empty label, park it in the catch-all and flag the source so
# it can be filtered out.
_orphan = final['rid_category'] == ''
if _orphan.any():
    print(f'WARNING: {int(_orphan.sum())} tickets came back with a code outside the taxonomy; '
          f'set to "{UNTAGGED}" with source=off_taxonomy')
    final.loc[_orphan, ['rid_category', 'rid_category_source']] = [UNTAGGED, 'off_taxonomy']

assert final['rid_category'].ne('').all(), 'a ticket ended up without a category'
assert set(final['rid_category']) <= set(CANONICAL), \
    f'labels outside the taxonomy: {set(final["rid_category"]) - set(CANONICAL)}'

final['rid_justification'] = final['rid_category_justification'].fillna('')
final.loc[_adjudicated_yes, 'rid_justification'] = \
    final.loc[_adjudicated_yes, 'rid_adjudication_justification'].fillna('')
final['rid_evidence'] = final['rid_category_evidence'].fillna('')
final.loc[_adjudicated_yes, 'rid_evidence'] = \
    final.loc[_adjudicated_yes, 'rid_adjudication_evidence'].fillna('')


In [ ]:
# ---- The funnel ---------------------------------------------------------
_is_bulk = final['rid_category'] == BULKID_LABEL
_n = len(final)
_rows = [
    ('Tickets in the export', _n),
    ('Regex on the whole thread (the Tableau figure)',
     int(final['rid_regex_flag_fulltext'].sum())),
    ('  ... matched only inside the agent reply, excluded',
     int((final['rid_regex_flag_fulltext'] & ~final['rid_regex_flag']).sum())),
    ('Regex on the customer turns -> sent to the adjudicator',
     int(final['rid_regex_flag'].sum())),
    (f'  ... adjudicated "{ADJ_YES}"', int(_adjudicated_yes.sum())),
    (f'  ... adjudicated "{ADJ_NO}", fall through to the taxonomy',
     int(final['rid_adjudication'].eq(ADJ_NO).sum())),
    ('Labelled BulkIDs by the taxonomy without the regex firing',
     int((_is_bulk & ~final['rid_regex_flag']).sum())),
    ('FINAL: BulkIDs / Registration IDs', int(_is_bulk.sum())),
]
print(f'{"":<62} {"tickets":>8} {"of all":>8}')
for _label, _v in _rows:
    print(f'{_label:<62} {_v:>8,} {_v / _n * 100:>7.1f}%')

print('\nWhere the rejected regex hits landed:')
_rej = final[final['rid_adjudication'].eq(ADJ_NO)]
print(_rej['rid_category'].value_counts().head(12).to_string())

print('\nLabel source:')
print(final['rid_category_source'].value_counts().to_string())


## 12. Write the deliverable


In [ ]:
XL_LIMIT = 32_767
MARKER = '  ...[TRUNCATED FOR EXCEL - see companion CSV for full text]'

OUT_COLS = [ID_COL, 'Incident_ID', 'Channel', DATE_COL, 'month',
            'rid_category', 'rid_category_source', 'rid_justification', 'rid_evidence',
            'rid_regex_flag', 'rid_regex_flag_fulltext', 'rid_regex_term',
            'rid_adjudication', 'rid_adjudication_justification', 'rid_adjudication_evidence',
            'rid_category_code', 'rid_customer_chars', 'rid_no_customer_turn',
            'conversation_chars_original',
            'was_truncated', TEXT_COL]
OUT_COLS = [c for c in OUT_COLS if c in final.columns]
deliverable = final[OUT_COLS].copy()
deliverable.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')
print(f'Wrote {FINAL_CSV.relative_to(BASE)} ({len(deliverable):,} rows, untruncated text)')


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if TEXT_COL in df.columns:
        long = df[TEXT_COL].astype(str).str.len() > XL_LIMIT
        df.loc[long, TEXT_COL] = (df.loc[long, TEXT_COL].astype(str)
                                  .str[:XL_LIMIT - len(MARKER)] + MARKER)
    return df


# -- Category summary -----------------------------------------------------
_vc = deliverable['rid_category'].value_counts()
category_summary = pd.DataFrame({
    'category': _vc.index, 'tickets': _vc.values,
    'share_%': (_vc.values / len(deliverable) * 100).round(2)})
category_summary['rule_assigned'] = category_summary['category'].map(
    deliverable[deliverable['rid_category_source'] == 'rule']
    ['rid_category'].value_counts()).fillna(0).astype(int)
category_summary['adjudicated'] = category_summary['category'].map(
    deliverable[deliverable['rid_category_source'] == 'adjudicated']
    ['rid_category'].value_counts()).fillna(0).astype(int)
category_summary['mean_conversation_chars'] = category_summary['category'].map(
    deliverable.groupby('rid_category')['conversation_chars_original'].mean().round(0))
category_summary['definition'] = category_summary['category'].map(CANONICAL)

by_month = pd.crosstab(deliverable['rid_category'], deliverable['month'])
by_channel = pd.crosstab(deliverable['rid_category'], deliverable['Channel'].fillna('(unknown)'))
regid_hits = deliverable[deliverable['rid_regex_flag']].copy()

# -- Taxonomy sheet -------------------------------------------------------
taxonomy_sheet = pd.DataFrame([
    {'module': MOD_T, 'code': CODE_OF_TAX[_l], 'label': _l, 'definition': _d,
     'raw_labels_folded_in': '; '.join(sorted(k for k, v in RAW_TO_CANONICAL.items() if v == _l)),
     'source': ('Distinct tab of ' + TAGGED_XLSX.name if _l in FROM_DISTINCT_TAB
                else 'Added: present in the raw labels, absent from the Distinct tab')}
    for _l, _d in CANONICAL.items()
] + [
    {'module': MOD_A, 'code': CODE_OF_ADJ[_l], 'label': _l, 'definition': _d,
     'raw_labels_folded_in': '', 'source': 'AUTHORED for this run'}
    for _l, _d in ADJ_DEFS.items()
] + [
    {'module': MOD_A, 'code': '', 'label': 'AUTHORED - adjudication rule',
     'definition': ADJ_RULE, 'raw_labels_folded_in': '',
     'source': 'AUTHORED for this run. Not from any specification. See notebook section 3.'},
    {'module': '(regex)', 'code': '', 'label': 'Tableau REGEXP_MATCH pattern',
     'definition': REGID_PATTERN, 'raw_labels_folded_in': '',
     'source': 'Ported verbatim from the Tableau calculated field on [Incident Subject].'},
])

# -- Method sheet ---------------------------------------------------------
_method = [
    ('Population',
     f'{len(deliverable):,} tickets from {TICKETS_XLSX.name}, dated '
     f'{deliverable[DATE_COL].min()} to {deliverable[DATE_COL].max()}. The filename says "last 3 '
     f'months"; the file actually spans {deliverable["month"].nunique()} months. Nothing was '
     f'filtered on date - every row in the export is classified.'),
    ('How BulkIDs tickets were found',
     'Two independent routes, either of which produces the label. (1) The Tableau '
     'Registration/Bulk ID regex applied to the CUSTOMER turns of the conversation, then confirmed '
     f'by an LLM adjudicator that never sees the agent reply. (2) The taxonomy module assigning '
     f'"{BULKID_LABEL}" on its own. Filter rid_category_source to separate them.'),
    ('Why the customer turns only',
     'The regex was written in Tableau against [Incident Subject]. This export has no subject '
     'column, only the full thread. Measured against the 7,082 manually tagged rows, the regex on '
     'full body text scores 97% recall and 31% precision - 356 of its 607 false positives are '
     'complimentary-access tickets where the AGENT answered by issuing registration IDs. '
     'Restricting to the customer turns removes that failure mode at source. The full-text flag '
     'is kept as rid_regex_flag_fulltext so the Tableau count stays reconcilable. The one '
     'exception is tickets with no customer turn at all - agent-logged notes and phone summaries '
     '- which are sent whole because the alternative is sending nothing; rid_no_customer_turn '
     'flags them and the adjudicator prompt tells the model to expect them.'),
    ('Where the taxonomy came from',
     f'The Distinct and Categories tabs of {TAGGED_XLSX.name} hold 25 hand-curated categories with '
     f'descriptions and synonyms; four of those rows are notes rather than categories and were '
     f'dropped. The categories carrying real volume among the 102 raw labels but absent from that '
     f'tab were added. All {len(RAW_TO_CANONICAL)} raw labels map onto the resulting '
     f'{len(CANONICAL)} categories with none left over - see docs/{TAX_MAP_MD.name} and the '
     f'Taxonomy sheet.'),
    ('Accuracy - what was and was not measured',
     f'A {GOLD_N}-row stratified sample of the manually tagged corpus was scored in notebook '
     f'section 9. Read that figure with its caveat: the tagged corpus Document column is mostly '
     f'the AGENT REPLY ALONE (1.8% carry a Customer: marker) where this data is a full thread. It '
     f'measures whether the cleaned categories are separable, not the accuracy of this run. No '
     f'number in this workbook is an accuracy figure for the production classification.'),
    ('How to get a real accuracy figure',
     'Hand-label a few hundred tickets sampled from the Tickets sheet, stratified on rid_category '
     'so the small categories are represented, and score against them.'),
    ('Rule-assigned rows',
     f'{int((deliverable["rid_category_source"] == "rule").sum()):,} tickets have under '
     f'{SHORT_CHAR_THRESHOLD} characters of payload and were set to "{UNTAGGED}" by rule with no '
     f'LLM call. Filter on rid_category_source to exclude them.'),
    ('Text truncation',
     f'{int(deliverable["was_truncated"].sum()):,} conversations exceeded {TRUNC_CAP:,} chars and '
     f'were sent as head {TRUNC_HEAD:,} + tail {TRUNC_TAIL:,} with a visible marker. The '
     f'Conversation column in the companion CSV holds the full text. Separately, '
     f'{int((deliverable["conversation_chars_original"] >= 32_767).sum()):,} conversations were '
     f'already cut at 32,767 chars in the source export and are incomplete at source.'),
    ('Speaker split',
     f'{int(final[TEXT_COL].str.contains(_SPEAKER_ANY).sum()):,} of {len(final):,} conversations '
     f'carry Customer:/Agent: markers, and the customer turns are about '
     f'{final["customer_text"].str.len().sum() / max(final[TEXT_COL].str.len().sum(), 1):.0%} of '
     f'the text by character. {int(final["rid_no_customer_turn"].sum()):,} conversations contain '
     f'no customer turn at all - agent-logged notes and phone summaries - and for those the whole '
     f'text is treated as customer text so they are not silently dropped from the regex '
     f'population. Column rid_no_customer_turn flags them.'),
    ('Authored content',
     'The adjudication rule is not from any specification - it was written for this run from the '
     'measured failure mode of the regex. It is printed in full on the Taxonomy sheet.'),
    ('Reproducing this',
     f'notebooks/16_reg_id_2026_bulkid_taxonomy.ipynb, run top to bottom on the .venv kernel. Prompt fingerprints: '
     + '; '.join(f'{k}={v}' for k, v in PROMPT_FINGERPRINTS.items())),
]
method_sheet = pd.DataFrame(_method, columns=['topic', 'detail'])

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as xw:
    fit_excel(deliverable).to_excel(xw, sheet_name='Tickets', index=False)
    category_summary.to_excel(xw, sheet_name='Category summary', index=False)
    fit_excel(regid_hits).to_excel(xw, sheet_name='RegID regex hits', index=False)
    by_month.to_excel(xw, sheet_name='Category x month')
    by_channel.to_excel(xw, sheet_name='Category x channel')
    taxonomy_sheet.to_excel(xw, sheet_name='Taxonomy', index=False)
    tax_map.to_excel(xw, sheet_name='Raw label map', index=False)
    method_sheet.to_excel(xw, sheet_name='Method', index=False)

print(f'Wrote {FINAL_XLSX.relative_to(BASE)}')
print(f'  Tickets           {len(deliverable):>6,} rows x {len(OUT_COLS)} cols')
print(f'  Category summary  {len(category_summary):>6} rows')
print(f'  RegID regex hits  {len(regid_hits):>6} rows')
print(f'  Taxonomy          {len(taxonomy_sheet):>6} rows')
print(f'  Raw label map     {len(tax_map):>6} rows')
print()
print(category_summary.drop(columns=['definition']).to_string(index=False))


## Afterwards

`generated/pipeline_config.json` now holds the two `regid_*` modules and nothing else. To put the
previous product's modules back, run the command in
`generated/_backup_pre_regid/RESTORE.txt` — the outputs of this run are already written to
`runs/output/` and do not depend on the config staying as it is.

**To re-run after editing a category definition:** change `CANONICAL` in section 2 and re-run from
section 3. The prompt fingerprint changes, the guard in section 7 discards the affected checkpoints,
and the next run pays for the records again rather than replaying stale answers. Editing only
`RAW_TO_CANONICAL` does not change the prompt, so a re-run there is free — but rebuild anyway, or
the deliverable's Raw label map sheet will disagree with the taxonomy.

**To widen the regex:** edit `REGID_PATTERN` in section 1. It is folded into the adjudicator's
fingerprint, so the adjudication checkpoint is discarded automatically. The self-test cell below it
is the place to add whatever new term prompted the change.
